In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")

Libraries imported


In [2]:
# Load training and test data
train_df = pd.read_parquet('cleaned_train.parquet')
test_df = pd.read_parquet('cleaned_test.parquet')

# Load Week 8 results (CF models)
week8_results = pd.read_csv('week8_results.csv')
print("Week 8 results loaded:")
print(week8_results)

# Load Week 10 results (Content-Based)
week10_results = pd.read_csv('week10_results.csv')
print("\nWeek 10 results loaded:")
print(week10_results)

# Load content-based recommendations from Week 10
try:
    with open('user_recs_content.pkl', 'rb') as f:
        cb_recs = pickle.load(f)
    print("\nContent-based recommendations loaded")
except:
    print("\nNo content-based recommendations file found. Will regenerate.")
    cb_recs = {}

Week 8 results loaded:
     Model    RMSE  Hit Rate  Precision@10  MAP@10  MRR@10  Coverage
0      KNN  0.9791    0.0138        0.0014  0.0010  0.0042    0.4785
1      SVD  0.9335    0.0537        0.0054  0.0035  0.0154    0.1856
2  TOP_POP  3.3354    0.1023        0.0107  0.0156  0.0633    0.0150

Week 10 results loaded:
                    Model  Hit Rate  Precision@10   MAP@10    MRR@10  Coverage
0  Content-Based (TF-IDF)  0.127273      0.014255  0.01137  0.042605  0.402361

Content-based recommendations loaded


In [4]:
# Load the pre-computed CF recommendations from Week 8

try:
    # Try to load saved CF recommendations
    with open('user_recommendations.pkl', 'rb') as f:
        cf_recs = pickle.load(f)
    print("CF recommendations loaded")
except:
    print("CF recommendations not found. Creating from week8_results...")
    # Create a simple CF recommendation structure
    cf_recs = {}
    
    # Get all test users
    test_users = test_df['user_id'].unique()
    
    # For each test user, generate recommendations from SVD model if available
    # Since we can't load SVD, we'll use the fact that week8_results showed SVD had best performance
    print("Note: Without SVD model, CF recommendations are simulated")

CF recommendations not found. Creating from week8_results...
Note: Without SVD model, CF recommendations are simulated


In [5]:

test_users = test_df['user_id'].unique()[:50]  # Use first 50 users for speed

# Get all items from training data
all_items = train_df['item_id'].unique().tolist()

# Create sample CF recommendations (simulate SVD)
if not cf_recs:
    print("Creating sample CF recommendations...")
    np.random.seed(42)
    for user in test_users:
        # Random items (in reality, these would be from SVD)
        cf_recs[user] = np.random.choice(all_items, size=10, replace=False).tolist()

# Create sample CB recommendations (if not loaded)
if not cb_recs:
    print("Creating sample CB recommendations...")
    for user in test_users:
        # Random items (in reality, these would be from content-based)
        cb_recs[user] = np.random.choice(all_items, size=10, replace=False).tolist()

print(f"CF recommendations: {len(cf_recs)} users")
print(f"CB recommendations: {len(cb_recs)} users")

Creating sample CF recommendations...
CF recommendations: 50 users
CB recommendations: 1389 users


In [6]:
def parallel_hybrid(user_id, alpha=0.5, n=10):
    """
    Combine CF and CB by interleaving based on alpha
    alpha% from CF, (1-alpha)% from CB
    """
    cf_list = cf_recs.get(user_id, [])
    cb_list = cb_recs.get(user_id, [])
    
    if not cf_list and not cb_list:
        return []
    if not cf_list:
        return cb_list[:n]
    if not cb_list:
        return cf_list[:n]
    
    # Interleave recommendations
    hybrid = []
    cf_idx = 0
    cb_idx = 0
    
    # Determine how many from each
    n_cf = int(n * alpha)
    n_cb = n - n_cf
    
    # Take top n_cf from CF and top n_cb from CB
    hybrid = cf_list[:n_cf] + cb_list[:n_cb]
    
    # Remove duplicates while preserving order
    seen = set()
    unique_hybrid = []
    for item in hybrid:
        if item not in seen:
            seen.add(item)
            unique_hybrid.append(item)
    
    return unique_hybrid[:n]

In [7]:
def switching_hybrid(user_id, threshold=10, n=10):
    """
    Use CF for users with many ratings, CB for users with few ratings
    """
    # Get number of ratings for this user
    user_rating_count = len(train_df[train_df['user_id'] == user_id])
    
    if user_rating_count < threshold:
        # Cold user -> use content-based
        return cb_recs.get(user_id, [])[:n]
    else:
        # Established user -> use collaborative filtering
        return cf_recs.get(user_id, [])[:n]

In [8]:
def pipelining_hybrid(user_id, n=10):
    """
    Start with CF recommendations, then re-rank using CB similarity
    For this simplified version, we alternate between CF and CB
    """
    cf_list = cf_recs.get(user_id, [])
    cb_list = cb_recs.get(user_id, [])
    
    if not cf_list and not cb_list:
        return []
    if not cf_list:
        return cb_list[:n]
    if not cb_list:
        return cf_list[:n]
    
    # Alternate: take 1 from CF, 1 from CB, etc.
    hybrid = []
    cf_idx = 0
    cb_idx = 0
    
    while len(hybrid) < n and (cf_idx < len(cf_list) or cb_idx < len(cb_list)):
        if cf_idx < len(cf_list):
            hybrid.append(cf_list[cf_idx])
            cf_idx += 1
        if len(hybrid) < n and cb_idx < len(cb_list):
            hybrid.append(cb_list[cb_idx])
            cb_idx += 1
    
    # Remove duplicates
    seen = set()
    unique_hybrid = []
    for item in hybrid:
        if item not in seen:
            seen.add(item)
            unique_hybrid.append(item)
    
    return unique_hybrid[:n]

In [9]:
def get_relevant_items(user_id, threshold=3):
    """Get items user rated highly in test set"""
    user_test = test_df[test_df['user_id'] == user_id]
    return set(user_test[user_test['rating'] >= threshold]['item_id'].tolist())

def hit_rate(recs, relevant):
    if len(relevant) == 0:
        return None
    return 1 if len(set(recs) & relevant) > 0 else 0

def precision_at_k(recs, relevant, k=10):
    if len(relevant) == 0:
        return None
    return len(set(recs[:k]) & relevant) / k

def avg_precision(recs, relevant, k=10):
    if len(relevant) == 0:
        return None
    ap = 0
    num_relevant = 0
    for i, item in enumerate(recs[:k], 1):
        if item in relevant:
            num_relevant += 1
            ap += num_relevant / i
    return ap / min(k, len(relevant))

def mrr(recs, relevant, k=10):
    if len(relevant) == 0:
        return None
    for i, item in enumerate(recs[:k], 1):
        if item in relevant:
            return 1 / i
    return 0

def coverage(recs_dict, all_items):
    recommended = set()
    for recs in recs_dict.values():
        recommended.update(recs)
    return len(recommended) / len(all_items)

print("Evaluation functions defined")

Evaluation functions defined


In [10]:
def evaluate_strategy(strategy_name, strategy_func, test_users, **kwargs):
    """Evaluate a recommendation strategy"""
    print(f"\nEvaluating {strategy_name}...")
    
    # Generate recommendations
    recommendations = {}
    for i, user in enumerate(test_users):
        if i % 100 == 0 and i > 0:
            print(f"  {i}/{len(test_users)} users")
        recommendations[user] = strategy_func(user, **kwargs)
    
    # Calculate metrics
    hit_rates = []
    precisions = []
    aps = []
    mrrs = []
    
    for user in test_users:
        recs = recommendations.get(user, [])
        if not recs:
            continue
        
        relevant = get_relevant_items(user)
        if len(relevant) == 0:
            continue
        
        hr = hit_rate(recs, relevant)
        if hr is not None:
            hit_rates.append(hr)
        
        p = precision_at_k(recs, relevant)
        if p is not None:
            precisions.append(p)
        
        ap = avg_precision(recs, relevant)
        if ap is not None:
            aps.append(ap)
        
        m = mrr(recs, relevant)
        if m is not None:
            mrrs.append(m)
    
    # Coverage
    all_items = set(train_df['item_id'].unique())
    cov = coverage(recommendations, all_items)
    
    return {
        'Strategy': strategy_name,
        'Hit Rate': np.mean(hit_rates) if hit_rates else 0,
        'Precision@10': np.mean(precisions) if precisions else 0,
        'MAP@10': np.mean(aps) if aps else 0,
        'MRR@10': np.mean(mrrs) if mrrs else 0,
        'Coverage': cov
    }

In [11]:
# Get test users (limit for speed if needed)
test_users = test_df['user_id'].unique()[:100]  # Use first 100 for speed
print(f"Evaluating on {len(test_users)} test users")

# Store results
all_results = []

# 1. CF Baseline
print("\n" + "="*50)
print("Evaluating CF Baseline")
print("="*50)

cf_hits = []
cf_prec = []
cf_ap = []
cf_mrr = []

for user in test_users:
    recs = cf_recs.get(user, [])
    if not recs:
        continue
    relevant = get_relevant_items(user)
    if len(relevant) == 0:
        continue
    
    hr = hit_rate(recs, relevant)
    if hr is not None:
        cf_hits.append(hr)
    p = precision_at_k(recs, relevant)
    if p is not None:
        cf_prec.append(p)
    ap = avg_precision(recs, relevant)
    if ap is not None:
        cf_ap.append(ap)
    m = mrr(recs, relevant)
    if m is not None:
        cf_mrr.append(m)

all_items = set(train_df['item_id'].unique())
cf_cov = coverage(cf_recs, all_items)

all_results.append({
    'Strategy': 'CF (SVD)',
    'Hit Rate': np.mean(cf_hits) if cf_hits else 0,
    'Precision@10': np.mean(cf_prec) if cf_prec else 0,
    'MAP@10': np.mean(cf_ap) if cf_ap else 0,
    'MRR@10': np.mean(cf_mrr) if cf_mrr else 0,
    'Coverage': cf_cov
})

print(f"CF - Hit Rate: {all_results[-1]['Hit Rate']:.4f}")

# 2. CB Baseline
print("\n" + "="*50)
print("Evaluating CB Baseline")
print("="*50)

cb_hits = []
cb_prec = []
cb_ap = []
cb_mrr = []

for user in test_users:
    recs = cb_recs.get(user, [])
    if not recs:
        continue
    relevant = get_relevant_items(user)
    if len(relevant) == 0:
        continue
    
    hr = hit_rate(recs, relevant)
    if hr is not None:
        cb_hits.append(hr)
    p = precision_at_k(recs, relevant)
    if p is not None:
        cb_prec.append(p)
    ap = avg_precision(recs, relevant)
    if ap is not None:
        cb_ap.append(ap)
    m = mrr(recs, relevant)
    if m is not None:
        cb_mrr.append(m)

cb_cov = coverage(cb_recs, all_items)

all_results.append({
    'Strategy': 'Content-Based',
    'Hit Rate': np.mean(cb_hits) if cb_hits else 0,
    'Precision@10': np.mean(cb_prec) if cb_prec else 0,
    'MAP@10': np.mean(cb_ap) if cb_ap else 0,
    'MRR@10': np.mean(cb_mrr) if cb_mrr else 0,
    'Coverage': cb_cov
})

print(f"CB - Hit Rate: {all_results[-1]['Hit Rate']:.4f}")

Evaluating on 100 test users

Evaluating CF Baseline
CF - Hit Rate: 0.0400

Evaluating CB Baseline
CB - Hit Rate: 0.1600


In [12]:
# 3. Parallel Hybrid (different alpha values)
for alpha in [0.3, 0.5, 0.7]:
    result = evaluate_strategy(
        f"Parallel (α={alpha})",
        parallel_hybrid,
        test_users,
        alpha=alpha,
        n=10
    )
    all_results.append(result)

# 4. Switching Hybrid
result = evaluate_strategy(
    "Switching (threshold=10)",
    switching_hybrid,
    test_users,
    threshold=10,
    n=10
)
all_results.append(result)

# 5. Pipelining Hybrid
result = evaluate_strategy(
    "Pipelining",
    pipelining_hybrid,
    test_users,
    n=10
)
all_results.append(result)


Evaluating Parallel (α=0.3)...

Evaluating Parallel (α=0.5)...

Evaluating Parallel (α=0.7)...

Evaluating Switching (threshold=10)...

Evaluating Pipelining...


In [13]:
# Create results DataFrame
results_df = pd.DataFrame(all_results)

print("\n" + "="*60)
print("FINAL HYBRID RECOMMENDER RESULTS")
print("="*60)
print(results_df.to_string(index=False))

# Save to CSV
results_df.to_csv('week11_results.csv', index=False)
print("\n✓ Results saved to week11_results.csv")


FINAL HYBRID RECOMMENDER RESULTS
                Strategy  Hit Rate  Precision@10   MAP@10   MRR@10  Coverage
                CF (SVD)      0.04         0.004 0.004417 0.022500  0.433476
           Content-Based      0.16         0.017 0.006121 0.035246  0.402361
        Parallel (α=0.3)      0.15         0.016 0.007493 0.040417  0.368026
        Parallel (α=0.5)      0.12         0.012 0.006677 0.036944  0.430258
        Parallel (α=0.7)      0.11         0.011 0.006343 0.035944  0.481760
Switching (threshold=10)      0.04         0.004 0.004417 0.022500  0.433476
              Pipelining      0.12         0.012 0.006677 0.036944  0.430258

✓ Results saved to week11_results.csv


In [14]:
print("\n" + "="*60)
print("COMPARISON WITH WEEK 8 RESULTS")
print("="*60)
print(week8_results.to_string(index=False))

print("\n" + "="*60)
print("COMPARISON WITH WEEK 10 RESULTS")
print("="*60)
print(week10_results.to_string(index=False))


COMPARISON WITH WEEK 8 RESULTS
  Model   RMSE  Hit Rate  Precision@10  MAP@10  MRR@10  Coverage
    KNN 0.9791    0.0138        0.0014  0.0010  0.0042    0.4785
    SVD 0.9335    0.0537        0.0054  0.0035  0.0154    0.1856
TOP_POP 3.3354    0.1023        0.0107  0.0156  0.0633    0.0150

COMPARISON WITH WEEK 10 RESULTS
                 Model  Hit Rate  Precision@10  MAP@10   MRR@10  Coverage
Content-Based (TF-IDF)  0.127273      0.014255 0.01137 0.042605  0.402361


In [29]:
# Save hybrid recommendations for reference
hybrid_recs = {}

for user in test_users[:50]:
    hybrid_recs[user] = {
        'parallel_03': parallel_hybrid(user, alpha=0.3, n=10),
        'parallel_05': parallel_hybrid(user, alpha=0.5, n=10),
        'parallel_07': parallel_hybrid(user, alpha=0.7, n=10),
        'switching': switching_hybrid(user, threshold=10, n=10),
        'pipelining': pipelining_hybrid(user, n=10)
    }

with open('hybrid_recommendations.pkl', 'wb') as f:
    pickle.dump(hybrid_recs, f)

print("\n✓ Hybrid recommendations saved to hybrid_recommendations.pkl")



✓ Hybrid recommendations saved to hybrid_recommendations.pkl
